In [ ]:

 #1. Dataset Gathering


In [ ]:
# --- GOOGLE COLAB SETUP ---
# 1. Upload your 1.7GB dataset ('Trade_DetailedTradeMatrix_E_All_Data_NOFLAG.csv') to your Google Drive.
# 2. Uncomment the code below to mount Google Drive to your Colab environment.

# from google.colab import drive
# drive.mount('/content/drive')

# 3. Set the file path below to point to the uploaded dataset in your Drive.
# For example: file_path = '/content/drive/MyDrive/Trade_DetailedTradeMatrix_E_All_Data_NOFLAG.csv'
file_path = 'Trade_DetailedTradeMatrix_E_All_Data_NOFLAG.csv' # Default local path

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_theme(style='darkgrid')

In [ ]:
#2. Loading and Pre-Processing (Raw Data EDA)


In [ ]:
chunk_size = 500000
chunks = []
target_crops = ['Wheat', 'Maize', 'Rice']

# Use regex to find elements containing our target crops
pattern = '|'.join(target_crops)

print("Loading large Data into Memory via chunks...")
for chunk in pd.read_csv(file_path, chunksize=chunk_size, encoding='latin1'):
    mask = chunk['Item'].str.contains(pattern, case=False, na=False)
    chunks.append(chunk[mask])

df_raw = pd.concat(chunks, ignore_index=True)
print('Raw Filtered Data Shape:', df_raw.shape)
df_raw.head()

In [ ]:
# ### Pre-processing Exploratory Data Analysis (EDA)


In [ ]:
# Display column info and data types
df_raw.info()

In [ ]:
# Check for Missing Values across Years
plt.figure(figsize=(14, 6))
sns.heatmap(df_raw.isnull(), cbar=False, cmap='YlGnBu')
plt.title('Missing Values in Raw Data (Dark = Missing)')
plt.show()

In [ ]:
# ## 3. Data Cleaning and Processing


In [ ]:
# 1. Filter irrelevant columns
cols_to_drop = ['Reporter Country Code', 'Reporter Country Code (M49)', 
                'Partner Country Code', 'Partner Country Code (M49)', 
                'Item Code', 'Item Code (CPC)', 'Element Code']
df_clean = df_raw.drop(columns=[col for col in cols_to_drop if col in df_raw.columns])

# 2. Reshape: Wide to Long format
year_cols = [col for col in df_clean.columns if col.startswith('Y') and col[1:].isdigit()]
id_cols = [col for col in df_clean.columns if col not in year_cols]

df_melted = pd.melt(df_clean, id_vars=id_cols, value_vars=year_cols, var_name='Year', value_name='Value')

# Convert 'Year' to integer instead of 'Y1986'
df_melted['Year'] = df_melted['Year'].str.replace('Y', '').astype(int)

# 3. Handle Missing Values (Drop rows where Value is missing as it means no trade occurred)
df_melted = df_melted.dropna(subset=['Value'])

# 4. Remove Duplicates
df_melted = df_melted.drop_duplicates()

# Optional: Quick check for any impossible negative physical quantities/values
df_melted = df_melted[df_melted['Value'] >= 0]

print("Cleaned and Melted Data Shape:", df_melted.shape)
df_melted.head()

In [ ]:
# ## 4. Post-Preprocessing EDA


In [ ]:
# 1. Trade Value Trends Over Time
plt.figure(figsize=(14, 6))
sns.lineplot(data=df_melted, x='Year', y='Value', hue='Element', estimator=sum, errorbar=None, marker="o")
plt.title('Global Agricultural Trade Over Time (Aggregated)')
plt.ylabel('Totals (Value/Quantity)')
plt.xlabel('Year')
plt.show()

In [ ]:
# 2. Distribution of Trade by Top Crops
plt.figure(figsize=(12, 6))
sns.barplot(data=df_melted, x='Item', y='Value', hue='Element', estimator=sum, errorbar=None)
plt.title('Total Trade Value & Quantity Distributed by Crop')
plt.ylabel('Sum of Values')
plt.xlabel('Crop Type')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Filter top 10 Reporter Countries by volume to see who exports/imports the most
top_reporters = df_melted.groupby('Reporter Countries')['Value'].sum().sort_values(ascending=False).head(10).index
df_top_reporters = df_melted[df_melted['Reporter Countries'].isin(top_reporters)]

plt.figure(figsize=(14, 7))
sns.boxplot(data=df_top_reporters, x='Reporter Countries', y='Value')
plt.yscale('log') # Log scale due to large outliers
plt.title('Distribution of Trade Values for Top 10 Reporter Countries (Log Scale)')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# ## 5. Exporting Preprocessed Data


In [ ]:
df_melted.to_csv('Preprocessed_Trade_Data.csv', index=False)
print("Successfully saved -> 'Preprocessed_Trade_Data.csv'")